# Cost crossover analysis

The study's central result: the distance from the MV network at which grid extension stops being the least-cost option, and how that distance varies with settlement size.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import matplotlib.pyplot as plt

from kajiado_lceo import load_config
from kajiado_lceo.pipeline import run

config = load_config("../config", scenario="baseline")
config.scenario

In [ ]:
from kajiado_lceo.sample import generate_sample_county

result = run(config, inputs=generate_sample_county(seed=42), write_outputs=False)
settlements = result.settlements
settlements[["households", "distance_to_mv_km", "lcoe_grid",
             "lcoe_minigrid", "lcoe_standalone",
             "least_cost_technology"]].head(10)

## LCOE against distance

Where the grid curve crosses the off-grid curves is the crossover. It is not a single number: it depends on settlement size, which is why the panels below split by household band.

In [ ]:
import numpy as np

bands = [(1, 5), (5, 50), (50, 500), (500, np.inf)]
fig, axes = plt.subplots(1, len(bands), figsize=(18, 4), sharey=True)
for ax, (low, high) in zip(axes, bands):
    block = settlements[(settlements["households"] >= low)
                        & (settlements["households"] < high)]
    for column, label in [("lcoe_grid", "Grid"),
                          ("lcoe_minigrid", "Mini-grid"),
                          ("lcoe_standalone", "Standalone")]:
        ax.scatter(block["distance_to_mv_km"], block[column], s=10,
                   alpha=0.6, label=label)
    ax.set_title(f"{low}-{high} households"); ax.set_yscale("log")
    ax.set_xlabel("Distance to MV (km)")
axes[0].set_ylabel("LCOE (USD/kWh)"); axes[0].legend()
plt.tight_layout(); plt.show()

## Estimating the crossover distance

For each band, the shortest distance at which grid ceases to be cheapest.

In [ ]:
for low, high in bands:
    block = settlements[(settlements["households"] >= low)
                        & (settlements["households"] < high)].copy()
    off_grid = block[["lcoe_minigrid", "lcoe_standalone"]].min(axis=1)
    loses = block[block["lcoe_grid"] > off_grid]
    crossover = loses["distance_to_mv_km"].min() if len(loses) else float("nan")
    print(f"{low}-{high} households: grid first loses at {crossover:.1f} km")